# ⚽ 5.3–5.4 Preprocessing Package による前処理 と Event Modeling Package によるイベント予測

[![Documentation Status](https://readthedocs.org/projects/openstarlab/badge/?version=latest)](https://openstarlab.readthedocs.io/en/latest/?badge=latest)
[![ArXiv](https://img.shields.io/badge/ArXiv-2502.02785-b31b1b?logo=arxiv)](https://arxiv.org/abs/2502.02785)
[![Discord](https://img.shields.io/badge/Discord-Join%20Chat-5865F2?logo=discord&logoColor=white)](https://discord.gg/yDrcywCs)

このノートブックでは、サッカーのイベントデータをモデリングするための効率的なパイプラインを提供します。データの前処理からトレーニング、評価、分析までの完全なワークフローを網羅しています。
![Football Icon](https://drive.google.com/uc?export=view&id=1qbYgNCQDJBvS1BsTFM2C4CPFeDhdOT-q)

（画像は [SoccerNet](https://www.soccer-net.org/) より）
完全なコードベースと関連ツールは [OpenSTARLab の GitHub](https://github.com/open-starlab) で公開されています。

* * *

## このノートブックの実行手順:

1. ⚙️ 必要なパッケージをインストールする
2. 📥 Wyscout イベントデータをダウンロードする
3. ⚽ `openstarlab-preprocessing` パッケージでデータを前処理する
4. ✂️ データセットを分割する
5. 🧠 `openstarlab-event` パッケージを用いてサッカーイベントモデルを学習するための YAMLを設定する
6. 🧠 `openstarlab-event` パッケージを用いてNMSTPP モデルを学習する  
   🔍 (オプション) [Optuna](https://github.com/optuna/optuna) を使用してハイパーパラメーターを最適化する
7. 📊 訓練済みモデルで推論を実行する
8. 📈 モデル予測結果を用いたパフォーマンス分析

## クレジット
*   Authors: Calvin Yeung & Keisuke Fujii
*   Affiliation: Nagoya University
*   Last updated: 2026-04-26
*   License: CC BY 4.0

# 📘 技術用語解説

**⚽ イベントデータ**

イベントデータとは、サッカーの試合中に発生するオンボールプレイヤー (ボールを保持しているプレイヤー) のアクションを構造化された時間順に記録したものです。各イベントは以下をキャプチャします。

- **何が起こったか** (例: パス、シュート、タックル)
- **いつ起こったか** (タイムスタンプまたは試合時間)
- **どこで起こったか** (フィールド座標)

イベントデータの典型的な属性には以下が含まれます。
- **タイムスタンプ**: 試合中にイベントが発生した時間。
- **アクションタイプ (マーカー)**: 特定のサッカーアクション (例: クロス、クリア、ドリブル)。
- **プレイヤーとチームの識別子**: アクションを実行した人物に関する情報。
- **開始位置 (x, y)**: ピッチ上のアクションの空間的な起点。
- **結果**: アクションが成功したか失敗したか。

イベントデータは試合のダイナミクスをモデリングおよび分析することを可能にし、研究者やアナリストがチームの行動を理解し、将来のアクションを予測し、プレイヤーの意思決定を評価するのに役立ちます。

* * *

**🔁 ポゼッション**

サッカーにおけるポゼッションとは、一方のチームがボールをコントロールし続ける期間を指します。ポゼッションシーケンスは通常、チームがボールを奪ったときに始まり、以下の理由でボールを失ったときに終了します。
- 相手によるインターセプトまたはタックル
- シュート
- ファウルまたは中断
- ボールがプレーエリア外に出る

ポゼッションを分析することで、チームの優位性、戦略的な意図、および得点機会を生み出す個人または集団のアクションの価値を定量化するのに役立ちます。

# 🔍 背景

サッカーにおけるイベントモデリングとは、パス、シュート、ドリブルなどの**ゲーム内のアクションのシーケンスからパターンを学習**し、時間の経過とともにゲームがどのように展開するかを理解および予測するタスクを指します。

モデリングの目的は、以前のイベントのシーケンス (時間 *t − n* から *t*) に基づいて、試合における次のイベント (時間 *t + 1*) を予測することです。各イベントには複数のコンポーネントが含まれます。いつ発生するか (イベント間時間)、ピッチ上のどこで発生するか (ゾーン)、およびどのようなタイプのアクションか (例: パス、シュート)。

正確なイベントモデリングにより、以下が可能になります。

* **戦術的な洞察**: チームがどのようにボールを進め、チャンスを作り出すかを理解する。
* **パフォーマンス評価**: 意思決定と実行の質を評価する。
* **シナリオシミュレーション**: 分析またはトレーニングのために将来のアクションを予測する。

# 1.⚙️ 必要なパッケージをインストールする

a) 開始する前に、Colab のランタイムを GPU に設定してください。
- **ランタイム > ランタイムタイプの変更 > ハードウェアアクセラレーター > GPU**

b) 以下のコードブロックを実行してください

In [1]:
# 1. Install Python 3.10 + pip quietly
!sudo apt-get update -qq > /dev/null 2>&1
!sudo apt-get install -y -qq python3.10 python3.10-distutils python3.10-venv curl > /dev/null 2>&1
!curl -sS https://bootstrap.pypa.io/get-pip.py | sudo python3.10 > /dev/null 2>&1

# 2. Set Python 3.10 as default quietly
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1 > /dev/null 2>&1
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 2 > /dev/null 2>&1
!sudo update-alternatives --set python3 /usr/bin/python3.10 > /dev/null 2>&1

# 3. Upgrade pip & install kernel quietly
!python3 -m pip install --upgrade pip setuptools wheel ipykernel -q > /dev/null 2>&1
!python3 -m ipykernel install --user --name=python310 --display-name "Python 3.10 (Colab)" > /dev/null 2>&1

# 4. Confirm python version 3.10.x
!python --version

Python 3.10.12


必要なパッケージがインストールされているか確認します

In [2]:
import os
import sys

# 5. Virtual environment
venv_path = '/content/openstarlab_env'
python_bin = os.path.join(venv_path, 'bin', 'python3')
pip_bin = os.path.join(venv_path, 'bin', 'pip')
site_packages_path = os.path.join(venv_path, 'lib', f'python3.10', 'site-packages')

if not os.path.exists(venv_path):
    !python3 -m venv {venv_path}
    print(f"✅ venv created at {venv_path}")
else:
    print(f"✅ venv exists at {venv_path}")

if site_packages_path not in sys.path:
    sys.path.insert(0, site_packages_path)
    print(f"✅ '{site_packages_path}' added to sys.path")
else:
    print("✅ venv already in sys.path")

print("\nsys.path[0]:", sys.path[0])

# 6. Upgrade pip
!{pip_bin} install --upgrade pip setuptools wheel -q

# 7. Install packages
!{pip_bin} install -q \
    numpy==1.24.4 \
    scipy==1.10.1 \
    pandas \
    torch==2.4.0+cu121 torchvision==0.19.0+cu121 torchaudio==2.4.0 --extra-index-url https://download.pytorch.org/whl/cu121 \
    openstarlab-preprocessing==0.1.35 \
    openstarlab-event==0.1.24 \
    gdown

print("✅ Packages installed")

✅ venv created at /content/openstarlab_env
✅ '/content/openstarlab_env/lib/python3.10/site-packages' added to sys.path

sys.path[0]: /content/openstarlab_env/lib/python3.10/site-packages
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 KB 17.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Packages installed


In [3]:
# 8. Confirm the installed packages
%%bash
# Write the Python script
cat << EOF > run.py

import numpy as np
import scipy as sp
import pandas as pd
print("numpy version: ",np.__version__) #v1.24.4
print("scipy version: ",sp.__version__) #v1.10.1
print("pandas version: ",pd.__version__)

try:
  import preprocessing
  print("preprocessing installed")
except:
  print("preprocessing not installed")

try:
  import event
  print("event installed")
except:
  print("event not installed")

try:
  import torch
  print("torch installed")
except:
  print("torch not installed")

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

numpy version:  1.24.4
scipy version:  1.10.1
pandas version:  2.3.3
preprocessing installed
event installed
torch installed


# 2.📥 Wyscout データをダウンロードする

ここでは、[Wyscout Open Access Dataset](https://doi.org/10.1038/s41597-019-0247-7) から提供されている 2017/2018 シーズンのサッカーイベントデータを利用します。このデータセットには、プレミアリーグ、ラ・リーガ、リーグ・アン、セリエA、ブンデスリーガの欧州主要5リーグの試合イベントが含まれています。これらの時空間的な試合イベントは、プレイヤーのアクションに関する詳細な洞察を提供し、サッカー分析におけるデータ駆動型モデルのトレーニングに不可欠です。

もし下記のPythonを通したダウンロードが失敗した場合でも、直接ダウンロードできる場合があります。その場合、eventとmatchesのフォルダを作成し、Download=Falseにすることで、再実行してみてください。
（2026/04/26現在、そのような状況なので、デフォルトをDownload=Falseにしています）

*参照: Pappalardo L, Cintia P, Rossi A et al. (2019). A public data set of spatio-temporal match events in soccer competitions. Scientific Data, 6(1):1–15.*

In [8]:
!mkdir -p './event'
!mkdir -p './matches'
# もしPythonを通したダウンロードが失敗する場合は、https://figshare.com/ndownloader/files/14464685/events.zipと
# https://figshare.com/ndownloader/files/14464622/matches.zipから直接ダウンロードし、./eventと./matchesにセットしてください。

In [9]:
%%bash
# Write the Python script
cat << EOF > run.py

import numpy as np
import pandas as pd
import subprocess
import requests
import os

def ensure_dir_exists(path):
    subprocess.run(['mkdir', '-p', path])

def download_with_requests(url, output_directory='.', filename=None):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers, stream=True)

    if response.status_code == 200:
        ensure_dir_exists(output_directory)

        if filename is None:
            filename = url.split('/')[-1]

        filepath = f"{output_directory}/{filename}"
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded {url} to {filepath} successfully.")
    else:
        print(f"Failed to download {url}. Status code: {response.status_code}")


def download_with_wget(url, output_directory='.'):
    # Construct the wget command
    command = ['wget', url, '-P', output_directory]

    # Execute the command
    try:
        subprocess.run(command, check=True)
        print(f"Downloaded {url} successfully.")
    except subprocess.CalledProcessError as e:
        print(f"Failed to download {url}. Error: {e}")

def download_wyscout_data(event_path,matches_path):
  # download the wyscout data
  event_url = "https://figshare.com/ndownloader/files/14464685/events.zip"
  matches_url = "https://figshare.com/ndownloader/files/14464622/matches.zip"


  #use either wget or requests to download the data
  # download_with_wget(event_url, event_path)
  # download_with_wget(matches_url, matches_path)

  download_with_requests(event_url, output_directory=event_path, filename="events.zip")
  download_with_requests(matches_url, output_directory=matches_path, filename="matches.zip")


# path for the wyscout data
event_path='./event'
matches_path='./matches'

# download wyscout open data
Download=False
if Download:
  download_wyscout_data(event_path,matches_path)
  print("Download completed")
else:
  print("Download skipped")



# unzip the downloaded files
subprocess.run(['unzip', 'event/events.zip', '-d', 'event'])
subprocess.run(['unzip', 'matches/matches.zip', '-d', 'matches'])

# remove the unnecessary files (expect England files)
subprocess.run(['rm', '-rf', 'event/events.zip'])
subprocess.run(['rm', '-rf', 'matches/matches.zip'])
subprocess.run(['rm', '-rf', 'event/events_European_Championship.json'])
subprocess.run(['rm', '-rf', 'event/events_World_Cup.json'])
subprocess.run(['rm', '-rf', 'matches/matches_European_Championship.json'])
subprocess.run(['rm', '-rf', 'matches/matches_World_Cup.json'])
subprocess.run(['rm', '-rf', 'event/events_France.json'])
subprocess.run(['rm', '-rf', 'matches/matches_France.json'])
subprocess.run(['rm', '-rf', 'event/events_Italy.json'])
subprocess.run(['rm', '-rf', 'matches/matches_Italy.json'])
subprocess.run(['rm', '-rf', 'event/events_Spain.json'])
subprocess.run(['rm', '-rf', 'matches/matches_Spain.json'])
subprocess.run(['rm', '-rf', 'event/events_Germany.json'])
subprocess.run(['rm', '-rf', 'matches/matches_Germany.json'])

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

Archive:  event/events.zip
  inflating: event/events_England.json  
  inflating: event/events_European_Championship.json  
  inflating: event/events_France.json  
  inflating: event/events_Germany.json  
  inflating: event/events_Italy.json  
  inflating: event/events_Spain.json  
  inflating: event/events_World_Cup.json  
Archive:  matches/matches.zip
  inflating: matches/matches_England.json  
  inflating: matches/matches_European_Championship.json  
  inflating: matches/matches_France.json  
  inflating: matches/matches_Germany.json  
  inflating: matches/matches_Italy.json  
  inflating: matches/matches_Spain.json  
  inflating: matches/matches_World_Cup.json  
Download skipped


# 3.⚽ openstarlab-preprocessing パッケージでデータを前処理する


このセクションでは、`openstarlab-preprocessing` パッケージを使用して生のイベントデータを前処理し、トレーニングセット、検証セット、テストセットに分割します。

**ステップ概要**

- イベントデータを読み込み、前処理する
- データをトレーニングセット、検証セット、テストセットに分割する

**サポートされているリソース**

- ✅ [サポートされているデータプロバイダーリスト](https://openstarlab.readthedocs.io/en/latest/Pre_Processing/Sports/Event_data/Data_Provider/index.html)
- 📄 [サポートされているデータ形式リスト](https://openstarlab.readthedocs.io/en/latest/Pre_Processing/Sports/Event_data/Data_Format/Football/index.html)

> **注:** パッケージはリストされているすべてのプロバイダーからのイベントデータの読み込みをサポートしていますが、データ形式変換は現在 **StatsBomb** および **Wyscout** 形式に最適化されています。

In [10]:
%%bash
# Write the Python script
cat << EOF > run.py

# import the event data class from the package
import os
from preprocessing import Event_data

event_path='./event'
matches_path='./matches'

# load and preprocess the data (increase max_workers for faster processing)
wyscout_df=Event_data(
  data_provider='wyscout',          # Data provider
  event_path=event_path,           # Path to the event files
  wyscout_matches_path=matches_path,     # Path to the matches files
  preprocess_method="UIED",          # desire data format
  max_workers=1                # number of cpu thread
).preprocessing()

wyscout_df.to_csv('data.csv',index=False)
wyscout_df.head(10)

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

Preprocessing data from wyscout with method UIED
Loading data from wyscout
Loaded data from wyscout
Preprocessed data from wyscout with method UIED


100%|██████████| 380/380 [09:58<00:00,  1.58s/it]


変換結果の確認を行います。
Pandas を使ってdata.csvを読み込み、基本的な妥当性チェックを行う例が示されています。

以下のようなコードにより、必須の情報が揃っているか、欠損値が存在しないかを確認します。

In [11]:
%%bash
# Write the Python script
cat << EOF > run.py
import pandas as pd

df = pd.read_csv("data.csv")

# 必須列の存在確認
required = {"action","start_x","start_y","seconds","delta_T","team"}
missing = required - set(df.columns)
assert not missing, f"Missing columns: {missing}"

# 欠損値チェック
assert df["start_x"].notna().all() and df["start_y"].notna().all()
assert df["seconds"].notna().all() and df["delta_T"].notna().all()

print(df)

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

        match_id  poss_id  team  ...  distance dist2goal  angle2goal
0        2499719        0  1609  ...    0.0000    0.0000      0.5000
1        2499719        0  1609  ...   27.3146   74.9101      2.8846
2        2499719        0  1609  ...   21.0989   54.1858      2.8225
3        2499719        0  1609  ...   17.0188   69.7279      2.9353
4        2499719        0  1609  ...   17.4938   69.0953      2.6828
...          ...      ...   ...  ...       ...       ...         ...
558087   2500098      266  1633  ...   49.6641   14.9968      1.6409
558088   2500098      266  1633  ...   19.0689   34.0000      1.5708
558089   2500098      266  1633  ...   37.0417   14.7000      3.1416
558090   2500098      266  1633  ...    0.0000    0.0000      0.5000
558091   2500098      266  1633  ...    0.0000    0.0000      0.5000

[558092 rows x 22 columns]


# 4.✂️  データセットを分割する

試合は時系列順にIDが割り当てられていることが多いため、これらの ID に基づいてデータセットを訓練データセット、検証データセット、テストデータセットに分割することで、将来のデータが訓練データセットに入ることを避けられます。

In [12]:
%%bash
# Write the Python script
cat << EOF > run.py

import pandas as pd

# split the data into train valid and test
Train_ratio=0.8
Valid_ratio=0.1
Test_ratio=0.1

Train_id=[]
Valid_id=[]
Test_id=[]

df = pd.read_csv('data.csv')
id_list=df.match_id.unique()
Train_id+=id_list[0:round(df.match_id.nunique()*Train_ratio)].tolist()
Valid_id+=id_list[round(df.match_id.nunique()*Train_ratio):round(df.match_id.nunique()*(Train_ratio+Valid_ratio))].tolist()
Test_id+=id_list[round(df.match_id.nunique()*(Train_ratio+Valid_ratio)):].tolist()

wyscout_df=pd.read_csv('data.csv')

train=wyscout_df[wyscout_df["match_id"].isin(Train_id)]
valid=wyscout_df[wyscout_df["match_id"].isin(Valid_id)]
test=wyscout_df[wyscout_df["match_id"].isin(Test_id)]

#print number of unique match in each split
print("Number of unique match in each split")
print("Train: ",train.match_id.nunique())
print("Valid: ",valid.match_id.nunique())
print("Test: ",test.match_id.nunique())

# edit the path of the datasets to desire location
train.to_csv("train.csv",index=False)
valid.to_csv("valid.csv",index=False)
test.to_csv("test.csv",index=False)
print("Datatset splitting done")

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

Number of unique match in each split
Train:  304
Valid:  38
Test:  38
Datatset splitting done


## 5.🛠️ openstarlab-event パッケージを使用して学習するためのYAMLファイルを設定する

このセクションでは、`openstarlab-event` パッケージと YAML 形式の設定ファイルを使用して、[**NMSTPP**](https://github.com/calvinyeungck/Football-Match-Event-Forecast) (Neural Marked Spatio-Temporal Point Process) モデルをトレーニングします。このファイルは、データセットのパスやハイパーパラメータなど、トレーニングの設定を指定します。

YAML ファイルは**2 つの方法**で簡単に変更できます。
- **プログラムによる方法**: Python スクリプトまたは Colab ノートブック内で `PyYAML` を使用して YAML ファイルを直接編集します。
- **手動による方法**: 任意の IDE またはテキストエディターで `train_NMSTPP.yaml` ファイルを開き、必要に応じて変更します。

🔧 `train_NMSTPP.yaml` の重要なパラメータ:
- `train_path`: トレーニング用 CSV ファイルへのパス
- `valid_path`: 検証用 CSV ファイルへのパス
- `save_path`: トレーニング済みモデルとログの保存先ディレクトリ
- `num_epoch`: トレーニングのエポック数
- `dataloader_num_worker`: データローダーが使用するワーカーの数 (Colab では 2 が推奨されます)
- `test`: テスト/デバッグモードのオプションフラグ (`True` または `False`)

以下は、Python を使用して YAML ファイルを直接変更し、モデルをトレーニングする例です。

In [13]:
!wget -q https://raw.githubusercontent.com/open-starlab/Event/main/event/sports/soccer/models/model_yaml/train_NMSTPP.yaml

In [14]:
%%bash
# Write the Python script
cat << EOF > run.py

import yaml

# Load YAML
with open('./train_NMSTPP.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Modify paths
config['train_path'] = './train.csv'
config['valid_path'] = './valid.csv'
config['save_path'] = './'
# Modify hyperparameters
config['num_epoch'] = 2
config['dataloader_num_worker'] = 2
# config['test']=False #Set to Ture on default for error testing

# Save updated YAML
with open('./train_NMSTPP.yaml', 'w') as f:
    yaml.dump(config, f)

# Check the modified file content
print("[YAML file Content]")

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

cat train_NMSTPP.yaml

[YAML file Content]
NN_action_num_layers: 2
NN_deltaT_num_layers: 1
NN_location_num_layers: 1
action_embedding_out_len: 9
basic_features:
- action
- delta_T
- start_x
- start_y
batch_size: 256
continuous_embedding_output_len: 14
dataloader_num_worker: 2
device: None
early_stop_patience: 3
eps: 1e-16
feature_embedding_output_len: 21
hidden_dim: 1024
learning_rate: 0.01
multihead_attention: 1
num_actions: 9
num_epoch: 2
optuna: false
optuna_n_trials: 100
other_features:
- team
- home_team
- success
- seconds
- deltaX
- deltaY
- distance
- dist2goal
- angle2goal
- home_score
- away_score
print_freq: 1
save_path: ./
scale_grad_by_freq: true
seq_len: 1
test: true
train_path: ./train.csv
use_other_features: true
valid_path: ./valid.csv


`Event_Model` クラスとイベントモデルのオプションの詳細については、[ドキュメント](https://openstarlab.readthedocs.io/en/latest/Event_Modeling/Sports/Soccer/model_class.html) を参照してください。

## 5.🧠 openstarlab-event パッケージを使用して NMSTPP モデルをトレーニングする

In [15]:
%%bash
# Write the Python script
cat << EOF > run.py

from event import Event_Model

# Initialize the NMSTPP model
model = Event_Model('NMSTPP', './train_NMSTPP.yaml')

model.train()

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

Epoch [1/2]
Batch [1/3] | Loss: 10.2900 | CEL_action: 2.0165 | RMSE_deltaT: 0.5140 | RMSE_location: 0.5703 | ACC_action: 0.5273 | F1_action: 0.0767 | MAE_deltaT: 3.4029 | MAE_x: 26.6886 | MAE_y: 29.9335
Batch [2/3] | Loss: 10.9163 | CEL_action: 1.9092 | RMSE_deltaT: 0.4516 | RMSE_location: 0.6749 | ACC_action: 0.5410 | F1_action: 0.0780 | MAE_deltaT: 3.2706 | MAE_x: 25.6270 | MAE_y: 44.8942
Batch [3/3] | Loss: 9.7869 | CEL_action: 1.8294 | RMSE_deltaT: 0.4289 | RMSE_location: 0.5813 | ACC_action: 0.5456 | F1_action: 0.0784 | MAE_deltaT: 3.2359 | MAE_x: 22.2361 | MAE_y: 38.8832
Training Loss: 9.7869
Batch [1/4] | Loss: 8.4631 | CEL_action: 1.3472 | RMSE_deltaT: 0.3568 | RMSE_location: 0.5332 | ACC_action: 0.6484 | F1_action: 0.0874 | MAE_deltaT: 3.0041 | MAE_x: 13.4131 | MAE_y: 40.8461
Batch [2/4] | Loss: 8.6644 | CEL_action: 1.4068 | RMSE_deltaT: 0.3595 | RMSE_location: 0.5460 | ACC_action: 0.6172 | F1_action: 0.0848 | MAE_deltaT: 3.2299 | MAE_x: 13.1761 | MAE_y: 42.8724
Batch [3/4] | 

# 6.🛠️ (オプション) Optuna を使用したハイパーパラメータ最適化

このセクションでは、強力なハイパーパラメータチューニングフレームワークである [Optuna](https://github.com/optuna/optuna) を使用して、**自動ハイパーパラメータ最適化**をオプションで有効にします。

以前と同様に、設定は同じ YAML ファイルを介して制御され、**2 つの方法**で編集できます。
- **プログラムによる方法**: Python コードを使用してパラメータを動的に設定または更新します。
- **手動による方法**: IDE または任意のテキストエディターで `train_NMSTPP_optuna.yaml` ファイルを開き、値を直接編集します。

Optuna 最適化を有効にするには、YAML 設定の関連フィールドを編集します。

⚙️ `train_NMSTPP_optuna.yaml` の Optuna 関連の主要パラメータ:
- `optuna`: Optuna 最適化を有効にするには `True` に設定します
- `optuna_n_trials`: 実行する試行回数 (例: `100`)

In [16]:
!wget -q https://raw.githubusercontent.com/open-starlab/Event/main/event/sports/soccer/models/model_yaml/train_NMSTPP_optuna.yaml

In [17]:
%%bash
# Write the Python script
cat << EOF > run.py

import yaml

# Load YAML
with open('./train_NMSTPP_optuna.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Modify paths
config['train_path'] = './train.csv'
config['valid_path'] = './valid.csv'
config['save_path'] = './'
# Modify hyperparameters
config['num_epoch'] = 2
config['dataloader_num_worker'] = 2
config['optuna_n_trials'] =2
# config['test']=False #Set to Ture on default for error testing

# Save updated YAML
with open('./train_NMSTPP_optuna.yaml', 'w') as f:
    yaml.dump(config, f)

# Check the modified file content
print("[YAML file Content]")

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

cat train_NMSTPP_optuna.yaml

[YAML file Content]
NN_action_num_layers:
- 1
- 2
- 3
NN_deltaT_num_layers:
- 1
- 2
- 3
NN_location_num_layers:
- 1
- 2
- 3
action_embedding_out_len:
- 9
basic_features:
- action
- delta_T
- start_x
- start_y
batch_size:
- 256
continuous_embedding_output_len:
- 14
dataloader_num_worker: 2
device: None
early_stop_patience: 5
eps:
- 1e-16
feature_embedding_output_len:
- 21
hidden_dim:
- 16
- 256
- 512
- 1024
- 2048
learning_rate:
- 0.01
multihead_attention:
- 1
num_actions: 9
num_epoch: 2
optuna: true
optuna_n_trials: 2
other_features:
- team
- home_team
- success
- seconds
- deltaX
- deltaY
- distance
- dist2goal
- angle2goal
- home_score
- away_score
print_freq: 1
save_path: ./
scale_grad_by_freq:
- true
seq_len: 40
test: true
train_path: ./train.csv
use_other_features: true
valid_path: ./valid.csv


In [18]:
%%bash
# Write the Python script
cat << EOF > run.py

from event import Event_Model

# Initialize the NMSTPP model
model = Event_Model('NMSTPP', './train_NMSTPP_optuna.yaml')

model.train()

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

Epoch [1/2]
Batch [1/3] | Loss: 17.6397 | CEL_action: 2.1891 | RMSE_deltaT: 0.7736 | RMSE_location: 1.1583 | ACC_action: 0.0312 | F1_action: 0.0068 | MAE_deltaT: 2.9470 | MAE_x: 66.5035 | MAE_y: 51.5593
Batch [2/3] | Loss: 14.5610 | CEL_action: 2.1162 | RMSE_deltaT: 0.7056 | RMSE_location: 0.8917 | ACC_action: 0.2734 | F1_action: 0.0483 | MAE_deltaT: 3.3884 | MAE_x: 47.5465 | MAE_y: 42.8064
Batch [3/3] | Loss: 12.4659 | CEL_action: 2.0314 | RMSE_deltaT: 0.6228 | RMSE_location: 0.7321 | ACC_action: 0.3581 | F1_action: 0.0578 | MAE_deltaT: 3.4738 | MAE_x: 36.6045 | MAE_y: 37.9094
Training Loss: 12.4659
Batch [1/4] | Loss: 8.0089 | CEL_action: 1.5364 | RMSE_deltaT: 0.3550 | RMSE_location: 0.4698 | ACC_action: 0.6484 | F1_action: 0.0874 | MAE_deltaT: 2.2623 | MAE_x: 18.5405 | MAE_y: 30.1924
Batch [2/4] | Loss: 8.0873 | CEL_action: 1.5644 | RMSE_deltaT: 0.3639 | RMSE_location: 0.4703 | ACC_action: 0.6172 | F1_action: 0.0848 | MAE_deltaT: 2.5460 | MAE_x: 18.9587 | MAE_y: 29.5015
Batch [3/4] 

[I 2026-04-26 01:23:08,384] A new study created in memory with name: no-name-b8409ebe-f6d2-45a5-bd15-a77f88d49122
[I 2026-04-26 01:23:15,718] Trial 0 finished with value: 7.163365602493286 and parameters: {'batch_size': 256, 'action_embedding_out_len': 9, 'scale_grad_by_freq': True, 'continuous_embedding_output_len': 14, 'hidden_dim': 256, 'feature_embedding_output_len': 21, 'NN_deltaT_num_layers': 3, 'NN_location_num_layers': 2, 'NN_action_num_layers': 3}. Best is trial 0 with value: 7.163365602493286.
[I 2026-04-26 01:23:27,770] Trial 1 finished with value: 7.574102878570557 and parameters: {'batch_size': 256, 'action_embedding_out_len': 9, 'scale_grad_by_freq': True, 'continuous_embedding_output_len': 14, 'hidden_dim': 512, 'feature_embedding_output_len': 21, 'NN_deltaT_num_layers': 2, 'NN_location_num_layers': 3, 'NN_action_num_layers': 1}. Best is trial 0 with value: 7.163365602493286.


# 7.📊 トレーニング済みモデルで推論を実行する

このセクションでは、トレーニング済みの NMSTPP モデルを使用して新しいデータに対する**推論**を実行する方法を示します。これは、検証セットまたはテストセットでのモデルパフォーマンスを評価する場合や、未知のデータに対する予測を生成する場合に役立ちます。

以下のいずれかを使用できます。
- **事前トレーニング済みモデル** (以下で自動的にダウンロードされます)
- 上記のトレーニングまたは Optuna チューニングステップで作成された独自の**トレーニング済みモデルチェックポイント**

以下では前者の例を示します。

In [19]:
%%bash
# Write the Python script
cat << EOF > run.py

import gdown

file_ids = [
    '1USCVhQzvaPuNM3njwAmG-KI3Rn_SZ9As',
    '1cVagVVvpcskN6Wc6DiINBu1zzXChBg-7',
    '1v3Zyl2if32b8zmcxxfbQYSUCvPc7JW-g'
]

for fid in file_ids:
    url = f'https://drive.google.com/uc?id={fid}'
    gdown.download(url, output=None, quiet=False)

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

Downloading...
From: https://drive.google.com/uc?id=1USCVhQzvaPuNM3njwAmG-KI3Rn_SZ9As
To: /content/_model_23.pth
100%|██████████| 492k/492k [00:00<00:00, 65.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1cVagVVvpcskN6Wc6DiINBu1zzXChBg-7
To: /content/hyperparameters.json
100%|██████████| 1.90k/1.90k [00:00<00:00, 5.71MB/s]
Downloading...
From: https://drive.google.com/uc?id=1v3Zyl2if32b8zmcxxfbQYSUCvPc7JW-g
To: /content/min_max_dict.json
100%|██████████| 661/661 [00:00<00:00, 2.08MB/s]


次に推論を行います。

📁 推論に必要なパラメータ:
- `model_path`: トレーニング済み `.pth` モデルファイルへのパス
- `model_config`: トレーニング中に使用された JSON 設定ファイルへのパス
- `valid_path`: 推論を実行したい CSV ファイルへのパス
- `min_max_dict_path`: 正規化に使用された辞書へのパス (出力の逆正規化に使用)
- `save_path`: 推論結果が保存されるディレクトリ

`inference()` 関数の詳細については、[ドキュメント](https://openstarlab.readthedocs.io/en/latest/Event_Modeling/Sports/Soccer/model_class.html) を参照してください。

In [20]:
%%bash
# Write the Python script
cat << EOF > run.py

from event import Event_Model

model = Event_Model('NMSTPP', './train_NMSTPP_optuna.yaml')

# Inference the validation set
# model_path='./out/optuna/20250603_032612/run_1/_model_1.pth' #example after training with the code above
# model_config='./out/optuna/20250603_032612/run_1/hyperparameters.json'
# inferenced_data,loss_df = model.inference(model_path, model_config)

# Inference on other dataset with pretrained model
model_path='./_model_23.pth'
model_config='./hyperparameters.json'
inference_data = './test.csv'
min_max_dict_path = './min_max_dict.json'
save_path = './'

inferenced_data, loss_df = model.inference(model_path, model_config, valid_path=inference_data, min_max_dict_path=min_max_dict_path, save_path=save_path)

print(loss_df)

# Save inferenced_data
inferenced_data.to_csv('inferenced_data.csv', index=False)
loss_df.to_csv('loss_df.csv', index=False)

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

   train_loss  CEL_action  RMSE_deltaT  ...  MAE_deltaT   MAE_x  MAE_y
0      4.6517      1.0064         0.24  ...      3.3089  6.8592  15.25

[1 rows x 9 columns]


/content/openstarlab_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
100%|██████████| 204/204 [02:32<00:00,  1.34it/s]


In [21]:
%%bash
# Write the Python script
cat << EOF > run.py

import pandas as pd

loss_df = pd.read_csv('loss_df.csv')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print(loss_df.to_string(index=False))

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

 train_loss  CEL_action  RMSE_deltaT  RMSE_location  ACC_action  F1_action  MAE_deltaT  MAE_x  MAE_y
     4.6517      1.0064         0.24         0.2445       0.669     0.1633      3.3089 6.8592  15.25


# 8.📈 モデルの予測結果を用いたパフォーマンス分析

推論実行後、効果的な攻撃系列を捉える**HPUS メトリック**を用いてさらに分析できます。

このステップでは、予測されたイベントを `match_id` でフィルタリングし、モデルのユーティリティ関数を使って HPUS を計算します。

モデルアプリケーションの詳細については、[ドキュメント](https://openstarlab.readthedocs.io/en/latest/Event_Modeling/Sports/Soccer/application.html)を参照してください。

# 🎯 包括的ポゼッション活用スコア (HPUS)

**包括的ポゼッション活用スコア (HPUS: Holistic Possession Utilization Score)** は、サッカーにおけるチームのポゼッションの**質と効率**を評価するために設計された指標です。これには、**アクションのタイプ**、それらの**フィールド上の位置**、およびそれらを実行する**時間**が考慮されます。

HPUS は、NMSTPP モデルからの予測 (イベント間時間（t）、場所（Zone）、行動（Action）タイプ) を使用して、ポゼッション内の各イベントに対して**包括的アクションスコア (HAS)** を計算します。

$$ 	{HAS} = \frac{\sqrt{E(\text{Zone}) \cdot E(\text{Action})}}{t} $$

ここで:
- $ t = 1 $ （イベント間時間が 1 より小さい場合）、それ以外の場合は実際のイベント間時間
- $ E(\text{Zone}) = 0 \cdot P(\text{Area}_0) + 5 \cdot P(\text{Area}_1) + 10 \cdot P(\text{Area}_2) $
- $ E(\text{Action}) = 0 \cdot P(\text{Possession~loss}) + 5 \cdot P(\text{Dribble, Pass}) + 10 \cdot P(\text{Cross, Shot}) $

HAS は [0, 10] の範囲にスケーリングされます。スコアが高いほど、より価値があり効率的なアクションを示します。

* * *

**🧮 HPUSの計算**

HPUS は、ポゼッション内のすべてのアクションにわたる HAS 値の**指数加重和**です。

$$ \text{HPUS} = \sum_{i=1}^{n} \phi(n + 1 - i) \cdot \text{HAS}_i
\quad \text{with} \quad \phi(x) = \exp(-0.3(x - 1)) $$

これにより、通常、最も決定的なアクション（例: シュートやクロス）である**最終アクション**に重点が置かれます。

* * *

**🔁 HPUS+**

**HPUS+** は、**攻撃的なアクション（シュートまたはクロス）で終了するポゼッションのみ**を考慮する亜種です。元論文（Yeung et al. 2025）では、このHPUS+の方が解釈しやすいとされています。

* * *
**📈 HPUS とパフォーマンス指標の相関**

**HPUS** は、**ゴール関連の入力を使用しないにもかかわらず**、主要なチームパフォーマンス指標と強い相関を示します。

- **チームの順位**: $ r = -0.77 $
- **得点**: $ r = 0.92 $
- **ゴール期待値 (xG)**: $ r = 0.92 $

これは、HPUS がポゼッションの質と効率を効果的に捉え、実際の試合結果とよく一致することを示唆しています。

* * *

**📌 まとめ**

- **HPUS** は、ゾーン、アクションタイプ、タイミングを使用して全体的なポゼッションの質を測定します。
- **HPUS+** は、攻撃につながったポゼッションを分離するため、攻撃効率の評価に役立ちます。

これらの指標は、チームがポゼッションをどのように意味のあるアクションに変換しているかを理解するための貴重な洞察を提供します。

次に、実際のHPUSの計算と可視化を行います。

In [23]:
%%bash
# Write the Python script
cat << EOF > run.py

import gdown
import pandas as pd
from event import Event_Model

model = Event_Model('NMSTPP', './train_NMSTPP_optuna.yaml')
save_path = './'

# Check if inferenced_data exist
try:
  inferenced_data = pd.read_csv('./inferenced_data.csv')
except NameError:
  fid = '1EA5AhQNDnKMMQ_NSppptmnT-l2Tv8Tas'
  url = f'https://drive.google.com/uc?id={fid}'
  gdown.download(url, output=None, quiet=False)
  pd.read_csv('./inferenced_data.csv')

match_id = inferenced_data.match_id.unique()
print(match_id)

# Perform quantitative analysis with the result summarized below
df_match = inferenced_data[inferenced_data.match_id == 2500028]
hpus = model.cal_HPUS(data = df_match, save_path = save_path, swap_home_away=True)
hpus_plus = model.cal_HPUS(data = df_match, save_path = save_path, plus = True, swap_home_away=True)

EOF
# Run the script using venv Python
unset MPLBACKEND
/content/openstarlab_env/bin/python run.py

[2500023 2500024 2500025 2500026 2500027 2500028 2500029 2500030 2500031
 2500032 2500033 2500034 2500035 2500036 2500037 2500038 2500039 2500040
 2500041 2500042 2500043 2500044 2500045 2500046 2500047 2500048 2500049
 2500050 2500051 2500052 2500053 2500054 2500055 2500056 2500057 2500058
 2500059 2500060]


/content/openstarlab_env/lib/python3.10/site-packages/event/sports/soccer/application/HPUS.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  HPUS_value = data.groupby(['match_id', 'poss_id']).apply(
/content/openstarlab_env/lib/python3.10/site-packages/event/sports/soccer/application/HPUS.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  HPUS_value = data.groupby(['match_id', 'poss_id']).apply(


**HPUS (包括的ポゼッション活用スコア)** メトリックは、サッカーにおけるポゼッション期間の**効果と効率**を評価するために設計された包括的なツールです。これには以下が組み込まれています。

- 📍 ピッチ上の**アクションの位置**
- ⚡ **アクションのタイプ** (例: パス、ドリブル、クロス、シュート)
- ⏱️ アクションを完了するのにかかる**時間**

HPUS は主に、シーケンスの戦術的および得点ポテンシャルを決定する上で最も影響力がある傾向がある、**ポゼッションの終盤に近いアクション**に重点を置いています。

**HPUS+** は、クロスやシュートのような**決定的な攻撃アクションで終了するポゼッションのみ**を含む、より焦点を絞ったバージョンです。これは**生産的でないポゼッションを除外**し、得点機会に直接貢献するシーケンスに焦点を当てます。

* * *

📊 **ケーススタディ**: ウェストハム・ユナイテッド vs マンチェスター・ユナイテッド (0 - 0)

- **大会**: プレミアリーグ 2017–2018
- **ホームチーム (ID 1633)**: ウェストハム・ユナイテッド
- **アウェイチーム (ID 1611)**: マンチェスター・ユナイテッド

* * *

![](https://drive.google.com/uc?export=view&id=1DXv2HnwLEzGsNQ7zWP9SUlPYU7LjlXqV)

🔷 **HPUS プロット**
- **HPUS (Holistic Possession Utilization Score)** は、すべてのチームポゼッションの全体的な効果を反映します。
- **X 軸**: 試合時間 (分)
- **Y 軸**: HPUS スコア
- マンチェスター・ユナイテッドは**強いポゼッションフェーズ**を示しました。
  - **15 分頃**、**30～35 分頃**、**65 分頃**、および**80 分頃**に HPUS 値が 8 を超え、最大 10.5 に達しました。
- ウェストハム・ユナイテッドは以下を示しました。
  - **10 分頃**に単一のピーク (HPUS ≈ 7.2)
  - 試合後半には散発的で中程度のポゼッション
- 全体的に、マンチェスター・ユナイテッドは**よりインパクトがあり、頻繁なポゼッション**を持っていました。

* * *
![](https://drive.google.com/uc?export=view&id=1B1q9pkh-I-Rypit6yGFCuEarq3MyCAjP)

🔶 **HPUS+ プロット**
- **HPUS+** は、**攻撃アクション (例: シュート、クロス) につながったポゼッション**を分離します。
- **X 軸**: 試合時間 (分)
- **Y 軸**: HPUS+ スコア
- マンチェスター・ユナイテッドはいくつかの**生産的な攻撃ポゼッション**を持っていました。
  - **15～40 分頃**、**65 分頃**、および**80～85 分頃**にピークがあり、値は最大 **3.7** でした。
- ウェストハム・ユナイテッドは**攻撃的なポゼッションがほとんどありません**でした。HPUS+ ラインは **0 付近で平坦**でした。

* * *

⚽ **試合概要**
- 最終スコア: **0–0**
- **マンチェスター・ユナイテッド**は、得点しなかったものの、**ポゼッションの質**と**攻撃的な意図**の点で優勢なチームでした。
- **ウェストハム・ユナイテッド**は、**低い HPUS+ 値**が示すように、ポゼッションを意味のある攻撃プレーに変換するのに苦労しました。

* * *

- マンチェスター・ユナイテッドは、**より多くのポゼッション (56%)**、**より多くのパス**、および**より多くの攻撃機会**を持ち、これは**HPUS および HPUS+ における優勢**と一致しています。
- 得点しなかったにもかかわらず、彼らの**6 回の枠内シュート**と**合計 16 回のシュート**は、彼らが**一貫して攻撃機会を創出**していたことを示しています。
- ウェストハムの限られた攻撃出力 (枠内シュート 2 回) は、彼らの**平坦な HPUS+ パフォーマンス**と一致しています。

# 📝 まとめ: OpenStarLab とその活用法

**OpenStarLab**（特に、このNotebookで示すEvent Modeling Package）は、深層学習を用いた試合イベントデータのモデリングと評価に焦点を当てた、高度なスポーツ分析のためのオープンソースフレームワークです。

**このチュートリアルでの主な機能:**
- **イベントデータ処理**: 複数のプロバイダーからのサッカーイベントデータを標準化します。
- **イベントモデリング**: 時間経過に伴うプレイヤーのアクションを追跡します。
- **パフォーマンス指標と分析**: ボール進行の効果と分析を測定します。

**活用法:**
- 研究者やアナリストが構造化されたメトリックを使用してパフォーマンスを評価するのに役立ちます。
- 試合結果とプレイヤーの意思決定を予測するための予測モデルの開発を可能にします。
- ポゼッションとアクションベースの洞察を通じて、チーム戦略 (例: ウェストハム対マンチェスター・ユナイテッド) を比較するのに役立ちます。

OpenStarLab は、生の試合データと深層学習ツールを使用した戦術的理解の間の架け橋となることを目指しています。